# Stage 4 - Portfolios

Question 3: does a risk-parity portfolio that uses the macro regimes do better, out of sample and net of costs,
than the same portfolio without regimes, and than 60/40?

- **Input:** `data/processed/` (from `02_clean.ipynb` and `03_regimes.ipynb`)
- **Output:** `data/processed/portfolio_returns.csv`, `portfolio_positions.csv`, `portfolio_corr_forecasts.csv`,
  `portfolio_performance.csv`

Run from top to bottom in a fresh kernel (about 15 seconds).

In [1]:
import numpy as np
import pandas as pd

import config
import helpers

## 1. Design, fixed before looking at the results

**Assets.** SPY (US stocks), TLT (long Treasuries), GLD (gold), DBC (commodities).

**Timing.** At each month-end `t` the portfolio is chosen with data known at `t` and held during month `t+1`.
First decision on 31 December 2006 (DBC only starts in February 2006), last on 31 July 2026:
236 months held, January 2007 to August 2026.

**Risk parity.** Each asset contributes the same share of the portfolio's risk (equal risk contribution).
The weights need a covariance matrix, built as volatilities x correlations:
- **volatilities:** last 63 trading days (3 months), the same for every strategy;
- **correlations:** the only thing that changes between strategies.

| Strategy | Correlation matrix used at `t` |
|---|---|
| `rp_unconditional` | average over all past months (since February 2006): the static benchmark |
| `rp_rolling` | average over the last 12 months: a benchmark that only looks at the market |
| `rp_quadrants` | average over the past months that followed the same quadrant as today (part 1 of stage 3) |
| `rp_gmm` | average for each regime, weighted by today's regime probabilities (main run of part 2b) |
| `rp_quadrants_revised`, `rp_gmm_revised` | the same two, with regimes from revised data (not available in real time) |
| `rp_oracle` | the correlation that actually happened next month: not investable, the best any forecast could do |

Plus `60_40`: 60% SPY, 40% TLT, rebalanced every month.

**Details.**
- Past correlations are measured month by month and then averaged, so every month counts the same,
  whatever its volatility (otherwise 2008 and March 2020 would dominate).
- The regime known at the end of month `s` is matched with the correlation of month `s+1`, exactly how it is used.
- A regime seen in fewer than 6 past months uses the unconditional correlation instead.
- **Leverage.** Positions are scaled so that predicted volatility is 10% per year, with at most 2x leverage.
  What is not invested earns the T-bill rate; what is borrowed costs the T-bill rate.
  So the regime matters twice: for the weights, and for the predicted risk, hence the leverage.
  Robustness: the same weights without leverage (fully invested).
- **Costs.** 10 basis points of every dollar traded. Sensitivity: 0 and 25.

**Measures.** Excess return over cash, volatility, Sharpe ratio, maximum drawdown, turnover, leverage,
and realised volatility against the 10% target: risk parity is a way to control risk, so a better correlation
forecast should mainly show up as realised risk closer to target. Sharpe ratios are compared with the test of
Ledoit and Wolf (2008), with standard errors robust to autocorrelation.
Periods: full sample, 2007-2016, 2017-2026, and the years 2008, 2020 and 2022.

In [2]:
ASSETS = ["SPY", "TLT", "GLD", "DBC"]
FIRST_DECISION = "2006-12-31"
CORR_START = "2006-02-01"   # all four assets from here: same history for every correlation estimate
VOL_DAYS = 63
ROLLING_MONTHS = 12
MIN_MONTHS = 6
TARGET_VOL = 0.10
MAX_LEVERAGE = 2.0
COST_BPS = 10
GMM_RUN = "12m_k3"          # main run of part 2b

PERIODS = {"full sample": ("2007", "2026"), "2007-2016": ("2007", "2016"), "2017-2026": ("2017", "2026"),
           "2008": ("2008", "2008"), "2020": ("2020", "2020"), "2022": ("2022", "2022")}

STRATEGIES = {   # name -> (correlation method, data used for the regimes)
    "rp_unconditional": ("unconditional", None),
    "rp_rolling": ("rolling", None),
    "rp_quadrants": ("quadrants", "realtime"),
    "rp_gmm": ("gmm", "realtime"),
    "rp_quadrants_revised": ("quadrants", "revised"),
    "rp_gmm_revised": ("gmm", "revised"),
    "rp_oracle": ("oracle", None),
}

## 2. Data

In [3]:
def load(name):
    return helpers.load_processed(name, index_col="date", parse_dates=True)


ret_d = load("returns_daily")[ASSETS].loc[CORR_START:]
ret_m = load("returns_monthly")[ASSETS]
rf = load("riskfree_monthly")["rf"]
quadrants = load("baseline_regimes")
probs = load("regimes_probabilities")

decisions = ret_m.loc[FIRST_DECISION:].index[:-1]
held = ret_m.loc[FIRST_DECISION:].index[1:]          # month t+1 for each decision t
print(f"{len(decisions)} decisions, {decisions[0].date()} -> {decisions[-1].date()};"
      f" held {held[0].date()} -> {held[-1].date()}")
assert ret_m.loc[held].notna().all().all() and rf.reindex(held).notna().all()

236 decisions, 2006-12-31 -> 2026-07-31; held 2007-01-31 -> 2026-08-31


### Correlation of each month, and the regime known just before it

In [4]:
corr_months, C = helpers.monthly_correlations(ret_d)
print(f"{len(C)} monthly correlation matrices, {corr_months[0].date()} -> {corr_months[-1].date()}")

before = corr_months - pd.offsets.MonthEnd(1)       # month-end at which the regime was known
signal = {}
for data in ["realtime", "revised"]:
    signal[("quadrants", data)] = quadrants[f"{data}_quadrant"].reindex(before).to_numpy()
    signal[("gmm", data)] = probs.filter(like=f"{data}_{GMM_RUN}_p").reindex(before).to_numpy()
    assert not pd.isna(signal[("quadrants", data)]).any() and not np.isnan(signal[("gmm", data)]).any()
print("regimes in the GMM run:", signal[("gmm", "realtime")].shape[1])

247 monthly correlation matrices, 2006-02-28 -> 2026-08-31
regimes in the GMM run: 3


## 3. Correlation forecasts and risk-parity weights

In [5]:
def forecast_corr(t, method, data):
    """Correlation matrix for month t+1, with data known at t. Returns (matrix, True if the fallback was used)."""
    past = corr_months <= t
    Cp = C[past]
    uncond = Cp.mean(axis=0)
    if method == "unconditional":
        return uncond, False
    if method == "rolling":
        return Cp[-ROLLING_MONTHS:].mean(axis=0), False
    if method == "oracle":
        return C[corr_months.get_loc(t + pd.offsets.MonthEnd(1))], False
    if method == "quadrants":
        same = signal[("quadrants", data)][past] == quadrants.loc[t, f"{data}_quadrant"]
        if same.sum() < MIN_MONTHS:
            return uncond, True
        return Cp[same].mean(axis=0), False
    # gmm: correlation of each regime (past months weighted by their probability), mixed with today's probabilities
    P = signal[("gmm", data)][past]
    today = probs.filter(like=f"{data}_{GMM_RUN}_p").loc[t].to_numpy()
    R, fallback = np.zeros_like(uncond), False
    for k in range(P.shape[1]):
        if P[:, k].sum() < MIN_MONTHS:
            Rk, fallback = uncond, fallback or today[k] > 0.5
        else:
            Rk = np.tensordot(P[:, k], Cp, axes=1) / P[:, k].sum()
        R += today[k] * Rk
    return R, fallback


# check on one date: every asset contributes 25% of the risk
cov = np.outer([0.15, 0.14, 0.16, 0.18], [0.15, 0.14, 0.16, 0.18]) * forecast_corr(decisions[0], "unconditional", None)[0]
w = helpers.erc_weights(cov)
print("test weights:", w.round(3), "| risk shares:", (w * (cov @ w) / (w @ cov @ w)).round(4))

test weights: [0.286 0.302 0.216 0.196] | risk shares: [0.25 0.25 0.25 0.25]


In [6]:
rows = []
for t in decisions:
    vol = ret_d.loc[:t].tail(VOL_DAYS).std().to_numpy() * np.sqrt(252)
    for name, (method, data) in STRATEGIES.items():
        R, fallback = forecast_corr(t, method, data)
        cov = np.outer(vol, vol) * R
        w = helpers.erc_weights(cov)
        risk_share = w * (cov @ w) / (w @ cov @ w)
        assert np.allclose(risk_share, 1 / len(w), atol=1e-6)
        predicted = np.sqrt(w @ cov @ w)
        rows.append({"date": t, "strategy": name, **dict(zip(ASSETS, w)), "predicted_vol": predicted,
                     "leverage": min(TARGET_VOL / predicted, MAX_LEVERAGE), "spy_tlt_corr": R[0, 1],
                     "fallback": fallback})
    rows.append({"date": t, "strategy": "60_40", "SPY": 0.6, "TLT": 0.4, "GLD": 0.0, "DBC": 0.0,
                 "predicted_vol": np.nan, "leverage": 1.0, "spy_tlt_corr": np.nan, "fallback": False})
positions = pd.DataFrame(rows).set_index(["strategy", "date"]).sort_index()
NAMES = ["60_40"] + list(STRATEGIES)

summary = positions.groupby("strategy")[ASSETS + ["leverage", "fallback"]].mean().loc[NAMES]
summary.round(2)

,SPY,TLT,GLD,DBC,leverage,fallback
strategy,,,,,,
60_40,0.60,0.40,0.00,0.00,1.00,0.00
rp_unconditional,0.27,0.35,0.19,0.19,1.36,0.00
rp_rolling,0.27,0.34,0.19,0.19,1.36,0.00
rp_quadrants,0.27,0.35,0.19,0.19,1.36,0.06
rp_gmm,0.28,0.35,0.19,0.19,1.36,0.04
rp_quadrants_revised,0.27,0.35,0.19,0.19,1.36,0.06
rp_gmm_revised,0.28,0.35,0.19,0.19,1.36,0.05
rp_oracle,0.27,0.34,0.19,0.19,1.43,0.00


How different is each portfolio from the static one, on average over the 236 decisions?

In [7]:
base = positions.loc["rp_unconditional"]
difference = pd.DataFrame({name: {
    "SPY-TLT correlation used": (positions.loc[name, "spy_tlt_corr"] - base["spy_tlt_corr"]).abs().mean(),
    "weights (sum of abs. differences)": (positions.loc[name, ASSETS] - base[ASSETS]).abs().sum(axis=1).mean(),
    "leverage": (positions.loc[name, "leverage"] - base["leverage"]).abs().mean(),
} for name in STRATEGIES if name != "rp_unconditional"}).T
difference.round(3)

,SPY-TLT correlation used,weights (sum of abs. differences),leverage
rp_rolling,0.183,0.079,0.098
rp_quadrants,0.056,0.033,0.033
rp_gmm,0.020,0.023,0.015
rp_quadrants_revised,0.059,0.036,0.031
rp_gmm_revised,0.029,0.017,0.019
rp_oracle,0.313,0.150,0.222


## 4. How good are the correlation forecasts?

Forecast of the SPY-TLT correlation for month `t+1` against the correlation that happened.

In [8]:
realised_next = pd.Series(C[:, 0, 1], index=corr_months).shift(-1).reindex(decisions)
corr_forecasts = positions["spy_tlt_corr"].unstack("strategy")[list(STRATEGIES)]
corr_forecasts["realised_next_month"] = realised_next

accuracy = {}
for name in STRATEGIES:
    if name == "rp_oracle":
        continue
    for period, (a, b) in PERIODS.items():
        f, r = corr_forecasts.loc[a:b, name], realised_next.loc[a:b]
        accuracy[(period, name)] = {"rmse": np.sqrt(((f - r) ** 2).mean()), "bias": (f - r).mean(),
                                    "corr with realised": f.corr(r)}
accuracy = pd.DataFrame(accuracy).T
accuracy.index.names = ["period", "strategy"]
accuracy.round(3)

,,rmse,bias,corr with realised
period,strategy,,,
full sample,rp_unconditional,0.383,-0.057,0.042
2007-2016,rp_unconditional,0.333,0.122,-0.004
2017-2026,rp_unconditional,0.429,-0.244,0.538
2008,rp_unconditional,0.357,0.322,-0.006
2020,rp_unconditional,0.252,0.003,-0.415
2022,rp_unconditional,0.467,-0.418,0.310
full sample,rp_rolling,0.312,-0.007,0.550
2007-2016,rp_rolling,0.305,0.015,0.157
2017-2026,rp_rolling,0.319,-0.030,0.528


## 5. Backtest

Positions chosen at `t` are held during `t+1`: return = sum of positions x asset returns + (1 - sum of positions) x cash.
During the month the positions drift with prices; the trade at the next rebalance is the gap between the drifted
and the new positions, and it pays the cost.

In [9]:
def strategy_positions(name, levered):
    p = positions.loc[name].copy()
    if levered and name != "60_40":
        p[ASSETS] = p[ASSETS].mul(p["leverage"], axis=0)
    return p


results = {}
for levered in [True, False]:
    for cost in [0, COST_BPS, 25]:
        for name in NAMES:
            pos = strategy_positions(name, levered)[ASSETS]
            results[(levered, cost, name)] = helpers.backtest(pos, ret_m, rf, cost)
print(len(results), "backtests")

48 backtests


## 6. Performance

In [10]:
def perf_table(levered, cost):
    rows = {}
    for period, (a, b) in PERIODS.items():
        for name in NAMES:
            rows[(period, name)] = helpers.performance(results[(levered, cost, name)].loc[a:b], rf)
    out = pd.DataFrame(rows).T
    out.index.names = ["period", "strategy"]
    return out


perf_main = perf_table(True, COST_BPS)
perf_main.loc["full sample"].round(3)

,excess return,volatility,sharpe,max drawdown,worst month,turnover per year,average exposure,months
strategy,,,,,,,,
60_40,0.069,0.104,0.663,-0.285,-0.107,0.317,1.000,236.0
rp_unconditional,0.073,0.107,0.684,-0.167,-0.081,1.921,1.356,236.0
rp_rolling,0.073,0.107,0.684,-0.177,-0.101,1.978,1.358,236.0
rp_quadrants,0.074,0.107,0.691,-0.165,-0.082,1.983,1.358,236.0
rp_gmm,0.073,0.107,0.682,-0.166,-0.081,1.955,1.364,236.0
rp_quadrants_revised,0.074,0.107,0.689,-0.169,-0.082,2.002,1.361,236.0
rp_gmm_revised,0.073,0.108,0.681,-0.167,-0.088,1.929,1.358,236.0
rp_oracle,0.100,0.112,0.890,-0.169,-0.091,4.432,1.434,236.0


In [11]:
for period in PERIODS:
    if period == "full sample":
        continue
    print(f"\n{period}:")
    print(perf_main.loc[period, ["excess return", "volatility", "sharpe", "max drawdown", "average exposure"]]
          .round(3).to_string())


2007-2016:
                      excess return  volatility  sharpe  max drawdown  average exposure
strategy                                                                               
60_40                         0.070       0.092   0.760        -0.285             1.000
rp_unconditional              0.065       0.101   0.641        -0.167             1.291
rp_rolling                    0.066       0.107   0.619        -0.177             1.367
rp_quadrants                  0.065       0.102   0.642        -0.165             1.304
rp_gmm                        0.065       0.102   0.641        -0.166             1.305
rp_quadrants_revised          0.066       0.102   0.648        -0.169             1.308
rp_gmm_revised                0.064       0.102   0.625        -0.167             1.291
rp_oracle                     0.083       0.113   0.740        -0.169             1.455

2017-2026:
                      excess return  volatility  sharpe  max drawdown  average exposure
strategy

### Is the difference in Sharpe ratio significant?

Each strategy against the static benchmark (`rp_unconditional`) and against `rp_rolling`.
Test of Ledoit and Wolf (2008): delta method with Newey-West standard errors.

In [12]:
excess = {name: results[(True, COST_BPS, name)]["net"] - rf.reindex(held).to_numpy() for name in NAMES}
tests = {}
for bench in ["rp_unconditional", "rp_rolling"]:
    for name in NAMES:
        if name != bench:
            tests[(bench, name)] = helpers.sharpe_diff_test(excess[name], excess[bench])
tests = pd.DataFrame(tests).T
tests.index.names = ["against", "strategy"]
tests.round(3)

sharpe difference     se      p
against          strategy                                             
rp_unconditional 60_40                            -0.022  0.193  0.910
                 rp_rolling                       -0.001  0.037  0.986
                 rp_quadrants                      0.007  0.010  0.501
                 rp_gmm                           -0.003  0.007  0.704
                 rp_quadrants_revised              0.004  0.012  0.735
                 rp_gmm_revised                   -0.004  0.008  0.625
                 rp_oracle                         0.206  0.059  0.001
rp_rolling       60_40                            -0.021  0.200  0.916
                 rp_unconditional                  0.001  0.037  0.986
                 rp_quadrants                      0.007  0.034  0.827
                 rp_gmm                           -0.002  0.036  0.957
                 rp_quadrants_revised              0.005  0.034  0.888
                 rp_gmm_revised                   -0.003  0.036  0.930
                 rp_oracle                         0.206  0.054  0.000

### Realised risk against the 10% target

In [13]:
realised_vol = perf_main["volatility"].unstack("strategy")[NAMES].loc[list(PERIODS)]
realised_vol.round(3)

strategy,60_40,rp_unconditional,rp_rolling,rp_quadrants,rp_gmm,rp_quadrants_revised,rp_gmm_revised,rp_oracle
period,,,,,,,,
full sample,0.104,0.107,0.107,0.107,0.107,0.107,0.108,0.112
2007-2016,0.092,0.101,0.107,0.102,0.102,0.102,0.102,0.113
2017-2026,0.116,0.113,0.108,0.112,0.114,0.113,0.114,0.112
2008,0.148,0.114,0.142,0.115,0.114,0.115,0.125,0.146
2020,0.140,0.135,0.148,0.137,0.134,0.138,0.132,0.146
2022,0.177,0.117,0.106,0.116,0.115,0.116,0.115,0.096


### Robustness: costs, and no leverage

In [14]:
cost_table = pd.DataFrame({f"{c} bps": perf_table(True, c).loc["full sample", "sharpe"] for c in [0, COST_BPS, 25]})
print("Sharpe ratio, full sample, by cost:")
print(cost_table.loc[NAMES].round(3).to_string())

perf_unlev = perf_table(False, COST_BPS)
print("\nwithout leverage (fully invested), full sample:")
print(perf_unlev.loc["full sample"].round(3).to_string())

Sharpe ratio, full sample, by cost:
                      0 bps  10 bps  25 bps
strategy                                   
60_40                 0.666   0.663   0.658
rp_unconditional      0.703   0.684   0.657
rp_rolling            0.702   0.684   0.656
rp_quadrants          0.710   0.691   0.663
rp_gmm                0.700   0.682   0.654
rp_quadrants_revised  0.707   0.689   0.661
rp_gmm_revised        0.699   0.681   0.654
rp_oracle             0.929   0.890   0.830

without leverage (fully invested), full sample:
                      excess return  volatility  sharpe  max drawdown  worst month  turnover per year  average exposure  months
strategy                                                                                                                       
60_40                         0.069       0.104   0.663        -0.285       -0.107              0.317               1.0   236.0
rp_unconditional              0.059       0.089   0.671        -0.177       -0.108         

## 7. Save

- `portfolio_returns.csv`: monthly net returns (10 bps), with and without leverage, and cash
- `portfolio_positions.csv`: at each decision, the risk-parity weights, leverage, predicted volatility and correlation
- `portfolio_corr_forecasts.csv`: forecast and realised SPY-TLT correlation for the next month
- `portfolio_performance.csv`: the performance table of section 6, with and without leverage

In [15]:
returns_out = pd.DataFrame({name: results[(True, COST_BPS, name)]["net"] for name in NAMES})
returns_out = returns_out.join(pd.DataFrame({f"{name}_unlevered": results[(False, COST_BPS, name)]["net"]
                                             for name in NAMES if name != "60_40"}))
returns_out["cash"] = rf.reindex(held).to_numpy()
perf_out = pd.concat({"levered": perf_main, "unlevered": perf_unlev}, names=["version"])

for df, name in [(returns_out.rename_axis("date"), "portfolio_returns"), (positions, "portfolio_positions"),
                 (corr_forecasts.rename_axis("date"), "portfolio_corr_forecasts"), (perf_out, "portfolio_performance")]:
    path = helpers.save_processed(df, name)
    print(f"saved {path.name}: {df.shape[0]} rows x {df.shape[1]} columns")

saved portfolio_returns.csv: 236 rows x 16 columns
saved portfolio_positions.csv: 1888 rows x 8 columns
saved portfolio_corr_forecasts.csv: 236 rows x 8 columns
saved portfolio_performance.csv: 96 rows x 8 columns
